# Patch Set
In this notebook, we will decided how to generate patchsets (a cache object for patch generation) from a slide.

In [1]:
from histokit.dataset.dataset import Dataset


cerv_mini_dataset = Dataset.from_index(
    "../data/icaird/cervical_mini/index.csv",
    "../data/icaird/cervical_mini/labels.json",
    slides_dir="slides",
    annotations_dir="annotations"
)

for sample in cerv_mini_dataset.samples():
    print(sample)

Sample(id='IC-CX-00001-01', slide_path=PosixPath('../data/icaird/cervical_mini/slides/IC-CX-00001-01.ome.tiff'), slide_schema=SlideSchema(kind='tiffslide', label_schema={'category': {'normal_inflammation': 0, 'low_grade': 1, 'high_grade': 2, 'malignant': 3}, 'subcategory': {'normal_inflammation': 0, 'cin1': 1, 'cin2': 2, 'cgin': 3, 'hpv': 4, 'cin3': 5, 'adenocarcinoma': 6, 'squamous_carcinoma': 7, 'other': 8}, 'split': {'train': 0, 'valid': 1, 'test': 2}}), annotation_path=PosixPath('../data/icaird/cervical_mini/annotations/IC-CX-00001-01.geojson'), annotation_schema=AnnotationSchema(kind='geojson', label_map={'background': 0, 'normal': 1, 'normal_inflammation': 2, 'low_grade': 3, 'high_grade': 4, 'malignant': 5}, cutout_label='normal', fill_label='normal', label_order=['background', 'normal_inflammation', 'low_grade', 'high_grade', 'malignant', 'normal']), metadata={'Unnamed: 0': 0, 'category': 'normal_inflammation', 'subcategory': 'normal_inflammation', 'split': 'train'})
Sample(id='

In [9]:
import datetime

import numpy as np
import pandas as pd

from histokit.dataset.sample import Sample
from histokit.io.annotations.annotation import AnnotationSet
from histokit.io.slides.slide import SlideBase
from histokit.patchset.context import PatchContext
from histokit.patchset.manifest import PatchSetManifest
from histokit.patchset.patchset import PatchSet
from histokit.segmentation.detector import TissueDetector
from histokit.segmentation.detectors import per_patch_canny_ranker
from histokit.utils.convert import to_frame_with_locations


def make_patchset_df(
    slide: SlideBase,
    patch_size: int,
    patch_level: int,
    tissue_detector: TissueDetector | None = None,
    annotation_set: AnnotationSet | None = None,
) -> pd.DataFrame:
    assert patch_level >= 0, "Patch level must be non-negative"
    assert slide.is_open, "Slide must be open to create patchset"

    # work out the size of the feature array
    size_in_patches = slide.size_in_patches(patch_size, patch_level)

    # if no tissue detector is provided, we assume the entire slide is foreground
    if tissue_detector is None:
        tissue_mask = np.ones(size_in_patches.as_shape(), dtype=bool)
    else:
        tissue_mask = tissue_detector(slide)

    # convert the tissue mask into dataframes with location information
    tissue_df = to_frame_with_locations(tissue_mask, 'tissue_score')

    # if there are annotations, render them as a grid and convert to a dataframe with location information
    if annotation_set is not None and len(annotation_set) > 0:
        annotation_labels = annotation_set.render_to_grid(size_in_patches.as_shape(), patch_size, patch_level)
        annotation_df = to_frame_with_locations(annotation_labels, 'annotation_label')
    
        # merge the tissue and annotation dataframes on the patch locations
        patch_df = tissue_df.merge(annotation_df, on=['row', 'column'], how='left')
    else:
        patch_df = tissue_df
        patch_df['annotation_label'] = np.nan  # add an empty annotation label column for consistency

    return patch_df

def make_patchset(sample: Sample, patch_size: int, patch_level: int, tissue_detector: TissueDetector | None = None) -> PatchSet:
    annotation_set = sample.make_annotations()  # will return None if there are no annotations, which is fine for make_patchset_df
    with sample.make_slide() as slide:
        patch_df = make_patchset_df(slide, patch_size, patch_level, tissue_detector, annotation_set)
    patch_df['context_id'] = 0  # for now we only have one context per patch, but this allows for multiple contexts in the future if needed
    context = PatchContext(sample=sample, level=patch_level, patch_size=patch_size)
    manifest = PatchSetManifest(created_at=str(datetime.datetime.now().isoformat()))
    return PatchSet(frame=patch_df, contexts=[context], manifest=manifest)

tissue_detector = per_patch_canny_ranker(224, 1)
for sample in cerv_mini_dataset.samples():
    patchset = make_patchset(sample, patch_size=224, patch_level=1, tissue_detector=tissue_detector)
    print(patchset.frame.head())

100%|██████████| 6125/6125 [00:00<00:00, 8404.21it/s]


   row  column  tissue_score  annotation_label  context_id
0    0       0           0.0               NaN           0
1    0       1           0.0               NaN           0
2    0       2           0.0               NaN           0
3    0       3           0.0               NaN           0
4    0       4           0.0               NaN           0


100%|██████████| 9880/9880 [00:01<00:00, 8884.92it/s]


   row  column  tissue_score  annotation_label  context_id
0    0       0           0.0                 1           0
1    0       1           0.0                 1           0
2    0       2           0.0                 1           0
3    0       3           0.0                 1           0
4    0       4           0.0                 1           0


100%|██████████| 8374/8374 [00:01<00:00, 6246.20it/s]

   row  column  tissue_score  annotation_label  context_id
0    0       0           0.0                 1           0
1    0       1           0.0                 1           0
2    0       2           0.0                 1           0
3    0       3           0.0                 1           0
4    0       4           0.0                 1           0
